# NB10 - Validation + leak audit

**Run once, after NB09.** Runs the **tiered ResNet-18 sniff test** (overall and per generator) plus structural checks, and prints a verdict. This is a *diagnostic tripwire*, not an automatic gate.

### How to read the sniff test (1-epoch real-vs-AI accuracy)
- **< 85%** - healthy. Clean real-vs-AI is genuinely hard; proceed.
- **85-95%** - investigate (could be a true AI signature, could be a mild leak).
- **> 95% almost immediately** - strong leak signal. Find and fix the shortcut before publishing.

Always look at the **per-generator** breakdown: if one model is trivially detectable while the rest are not, that model's branch has a localized problem.

**GPU: ON.** Internet ON. `SAMPLE_PER_CLASS` controls cost (raise it for a stronger test).

In [ ]:
import sys, subprocess
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs], check=True)
# diffusers stack + hub/io. sentencepiece+protobuf are needed by the T5 text encoders (Flux/PixArt).
pip("diffusers>=0.30", "transformers>=4.40", "accelerate", "safetensors",
    "sentencepiece", "protobuf", "huggingface_hub>=0.23", "pyarrow", "pillow", "torch", "torchvision", "scikit-learn")
print("deps installed")

In [ ]:
REPO_ID = "Shanmuk4622/ai-image-detection-dataset"
SAMPLE_PER_CLASS = 2500     # reals vs AI per sniff test; raise for a more sensitive probe
EPOCHS = 1
import os
from huggingface_hub import hf_hub_download
def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t: return t.strip()
    except Exception: pass
    for k in ("HF_TOKEN","HUGGINGFACE_TOKEN","HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k): return os.environ[k].strip()
    import getpass; return getpass.getpass("HF write token: ").strip()
TOKEN = load_token()
lib = hf_hub_download(REPO_ID, "ai_detect_common.py", repo_type="dataset", token=TOKEN)
import importlib.util
spec = importlib.util.spec_from_file_location("ai_detect_common", lib)
C = importlib.util.module_from_spec(spec); spec.loader.exec_module(C)
cfg = C.read_config(REPO_ID, TOKEN)
import torch
print("device:", "cuda" if torch.cuda.is_available() else "cpu")
assert torch.cuda.is_available(), "Turn the GPU ON for NB10."

In [ ]:
# helper: pull up to N decoded images (as 224x224 tensors) from a prefix
import io, numpy as np, torch
from PIL import Image

def load_images(prefix, n):
    out = []
    for f in C.list_shards(REPO_ID, prefix, TOKEN):
        loc = C.hf_hub_download(REPO_ID, f, repo_type="dataset", token=TOKEN)
        t = C.pq.read_table(loc, columns=["image"])
        for b in t.column("image").to_pylist():
            im = Image.open(io.BytesIO(b)).convert("RGB").resize((224, 224), Image.BILINEAR)
            out.append(np.asarray(im, dtype=np.float32) / 255.0)
            if len(out) >= n: return out
    return out

# same 224 resize is applied to real and AI alike, so it introduces no real-vs-AI asymmetry
print("loading reals ...")
reals = load_images("real/", SAMPLE_PER_CLASS)
print("real images:", len(reals))

In [ ]:
# tiny trainer for the sniff test
import torch, numpy as np
import torch.nn as nn
from torchvision.models import resnet18
from sklearn.model_selection import train_test_split

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)

def to_batch(arrs):
    x = torch.from_numpy(np.stack(arrs)).permute(0,3,1,2)   # N,3,224,224
    return (x - MEAN) / STD

def sniff(pos_imgs, neg_imgs, epochs=1, tag=""):
    """pos = AI (label 1), neg = real (label 0). Returns held-out accuracy."""
    n = min(len(pos_imgs), len(neg_imgs))
    if n < 50:
        print(f"  [{tag}] too few images ({n}); skipped"); return None
    X = pos_imgs[:n] + neg_imgs[:n]
    y = [1]*n + [0]*n
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
    dev = "cuda"
    net = resnet18(weights="IMAGENET1K_V1"); net.fc = nn.Linear(net.fc.in_features, 2); net.to(dev)
    opt = torch.optim.Adam(net.parameters(), lr=1e-4); lossf = nn.CrossEntropyLoss()
    Xtr_t, ytr_t = to_batch(Xtr).to(dev), torch.tensor(ytr).to(dev)
    bs = 64
    net.train()
    for ep in range(epochs):
        perm = torch.randperm(len(Xtr_t))
        for i in range(0, len(Xtr_t), bs):
            idx = perm[i:i+bs]
            opt.zero_grad()
            loss = lossf(net(Xtr_t[idx]), ytr_t[idx]); loss.backward(); opt.step()
    net.eval()
    Xte_t = to_batch(Xte).to(dev)
    with torch.no_grad():
        pred = net(Xte_t).argmax(1).cpu().numpy()
    acc = float((pred == np.array(yte)).mean())
    verdict = "healthy (<85)" if acc < 0.85 else ("INVESTIGATE (85-95)" if acc < 0.95 else "LEAK? (>95)")
    print(f"  [{tag}] held-out accuracy: {acc:.3f}  ->  {verdict}")
    return acc

print("trainer ready.")

In [ ]:
# per-generator sniff + overall
results = {}
for m in cfg["generators"]:
    ai = load_images(f"data/{m}/", SAMPLE_PER_CLASS)
    if not ai:
        print(f"[{m}] no images yet; skipping"); continue
    results[m] = sniff(ai, reals, epochs=EPOCHS, tag=m)

# overall: AI pooled across generators (round-robin) vs reals
import itertools
pools = [load_images(f"data/{m}/", max(1, SAMPLE_PER_CLASS // max(1,len(cfg['generators'])))) for m in cfg["generators"]]
ai_mixed = list(itertools.chain.from_iterable(pools))[:SAMPLE_PER_CLASS]
results["__overall__"] = sniff(ai_mixed, reals, epochs=EPOCHS, tag="overall")
print("\nsummary:", {k: (round(v,3) if v is not None else None) for k,v in results.items()})

In [ ]:
# structural checks + write a report
import json, tempfile, os, random
problems, warns = [], []

# split disjointness
sp = C.hf_hub_download(REPO_ID, "splits.parquet", repo_type="dataset", token=TOKEN)
spt = C.pq.read_table(sp)
from collections import Counter
multi = [k for k,v in Counter(spt.column("source_real_id").to_pylist()).items() if v>1]
if multi: problems.append(f"{len(multi)} source_real_id(s) in multiple splits")

# sample canonical check across real + each generator
def sample_canon(prefix, k=60):
    shards = C.list_shards(REPO_ID, prefix, TOKEN)
    if not shards: return 0,0
    loc = C.hf_hub_download(REPO_ID, shards[-1], repo_type="dataset", token=TOKEN)
    t = C.pq.read_table(loc, columns=["image"]); m=t.num_rows
    idx = range(m) if m<=k else random.sample(range(m), k)
    bad=0
    for i in idx:
        ok,_ = C.png_is_canonical(t.column("image")[i].as_py());  bad += (0 if ok else 1)
    return len(list(idx)), bad
for p in ["real/"] + [f"data/{m}/" for m in cfg["generators"]]:
    c,bad = sample_canon(p)
    if bad: problems.append(f"{p}: {bad}/{c} sampled not canonical")
    print(f"canonical sample {p}: {c-bad}/{c} ok")

report = {"sniff_test": {k:(round(v,3) if v is not None else None) for k,v in results.items()},
          "thresholds": {"healthy":"<0.85","investigate":"0.85-0.95","leak":">0.95"},
          "problems": problems, "warnings": warns}
tmp = os.path.join(tempfile.gettempdir(), "validation_report.json")
json.dump(report, open(tmp,"w"), indent=2)
C.HfApi(token=TOKEN).upload_file(path_or_fileobj=tmp, path_in_repo="validation_report.json",
        repo_id=REPO_ID, repo_type="dataset", commit_message="validation report")
print("\nRESULT:", "PASS (no structural problems)" if not problems else "REVIEW PROBLEMS ABOVE")
print(json.dumps(report, indent=2))